# Fine-tune **GuwenBERT** cho NER cổ văn 文言文 — Đồ án HCH
Mục tiêu: **phương pháp NER tối ưu** (học sâu, cùng miền 二十四史) + **output để so sánh** với rule/Hybrid.

**Chạy trên Colab GPU:** `Runtime → Change runtime type → T4 GPU`, rồi `Runtime → Run all`.

### ✅ Bản này đã sửa lỗi "train nhầm trên sample"
- **Tự tải dữ liệu thật (C-CLUE)** — không cần upload/đăng ký gì. Chạy là có data ngay.
- **GUARD**: nếu số câu train < ngưỡng → **báo lỗi dừng**, không cho train ra model rác.

### Dữ liệu & nhãn
- **C-CLUE** (mặc định): NER trên 二十四史, tải tự do — nhãn **PER, LOC, ORG, JOB(官职→TITLE)** → phủ **4/6 nhãn** bằng học sâu. Repo: https://github.com/jizijing/C-CLUE
- **GuNER2023** (tuỳ chọn): cần đăng ký https://guner2023.pkudh.org/ — nhãn PER/OFI(→TITLE)/BOOK.
- Nhãn **TME, NUM** không có trong 2 bộ này → vẫn lấy từ **rule/Hybrid** (cell 9 ghép lại đủ 6 nhãn).
- ⚠️ **C-CLUE là GIẢN THỂ**, data của bạn **PHỒN THỂ** → khi suy luận sẽ tự **OpenCC 繁→简** (căn lề 1-1, span vẫn lấy trên chữ phồn thể gốc).


## 0. Cài đặt

In [ ]:
!pip -q install "transformers>=4.40" "datasets>=2.19" "evaluate>=0.4" seqeval accelerate opencc-python-reimplemented regex

## 1. Cấu hình

In [ ]:
import os, re, json, csv, subprocess, numpy as np, torch
from pathlib import Path
from collections import Counter
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          TrainingArguments, Trainer, DataCollatorForTokenClassification)
from datasets import Dataset
import evaluate

DATASET  = "cclue"                 # "cclue" (tự tải, KHUYẾN NGHỊ) | "guner" (cần đăng ký)
MODEL_ID = "ethanyt/guwenbert-base"
MAX_LEN  = 256
MIN_TRAIN_SENTS = 50               # GUARD: dưới mức này -> báo lỗi, KHÔNG train

# nhãn dataset -> nhãn đề bài HCH
LABEL2PROJ = {"PER":"PER","LOC":"LOC","ORG":"ORG",
              "JOB":"TITLE","OFI":"TITLE","OFFICE":"TITLE",
              "BOOK":"BOOK","TIME":"TME","NUM":"NUM"}

# C-CLUE giản thể, data HCH phồn thể -> convert 繁->简 KHI SUY LUẬN
INFER_T2S = (DATASET == "cclue")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| dataset:", DATASET)

_cc = None
def to_simp(s):
    """繁->简 theo từng ký tự (giữ căn lề 1-1)."""
    global _cc
    if _cc is None:
        import opencc; _cc = opencc.OpenCC("t2s")
    out = []
    for c in s:
        r = _cc.convert(c); out.append(r if len(r) == 1 else c)
    return "".join(out)

## 2. Nạp dữ liệu THẬT
`cclue`: tự `git clone` repo rồi đọc `data_ner/{source,target}.txt` (ký tự ↔ nhãn, cách nhau bởi dấu cách).
`guner`: bỏ chú thích phần upload để nạp file `{thực thể|NHÃN}`.

In [ ]:
def remap(tag):
    if tag == "O": return "O"
    pre, lab = tag.split("-", 1)
    pl = LABEL2PROJ.get(lab.upper())
    return f"{pre}-{pl}" if pl else "O"          # nhãn không map -> O

parsed = []

if DATASET == "cclue":
    if not os.path.isdir("C-CLUE"):
        subprocess.run(["git","clone","--depth","1","https://github.com/jizijing/C-CLUE.git"], check=True)
    def read_pairs(src, tgt):
        S = [l.rstrip("\n") for l in open(src, encoding="utf-8")]
        T = [l.rstrip("\n") for l in open(tgt, encoding="utf-8")]
        data = []
        for s, t in zip(S, T):
            ch, tg = s.split(), t.split()
            if ch and len(ch) == len(tg):
                data.append((ch, [remap(x) for x in tg]))
        return data
    D = "C-CLUE/data_ner"
    parsed = read_pairs(f"{D}/source.txt", f"{D}/target.txt")
    parsed += read_pairs(f"{D}/dev.txt", f"{D}/dev-label.txt")

elif DATASET == "guner":
    def parse_guner(text):
        chars, tags = [], []; i, n = 0, len(text)
        while i < n:
            c = text[i]
            if c == "{":
                j = text.find("|", i); k = text.find("}", i)
                if j == -1 or k == -1 or j > k:
                    if not c.isspace(): chars.append(c); tags.append("O")
                    i += 1; continue
                ent, lab = text[i+1:j], text[j+1:k]; first = True
                for ch in ent:
                    if ch.isspace(): continue
                    chars.append(ch); tags.append(remap(("B-" if first else "I-")+lab)); first = False
                i = k + 1
            else:
                if not c.isspace(): chars.append(c); tags.append("O")
                i += 1
        return chars, tags
    from google.colab import files
    print("Upload file GuNER2023 (mỗi dòng 1 đoạn có markup {..|..})…")
    up = files.upload()
    for l in Path(next(iter(up))).read_text(encoding="utf-8").splitlines():
        if l.strip():
            ch, tg = parse_guner(l)
            if ch: parsed.append((ch, tg))

# ---- GUARD: không bao giờ train trên dữ liệu quá ít ----
n_ent = sum(1 for _, tg in parsed for t in tg if t != "O")
print(f"Đã nạp {len(parsed)} câu, {n_ent} thực thể.")
assert len(parsed) >= MIN_TRAIN_SENTS, (
    f"CHỈ CÓ {len(parsed)} câu (< {MIN_TRAIN_SENTS}). Dừng để tránh train ra model RÁC. "
    f"Kiểm tra lại bước tải dữ liệu.")
print("Nhãn (đã map về đề bài):", sorted({t for _, tg in parsed for t in tg if t != 'O'}))

### 2b. Chẩn đoán 繁/简 — tỉ lệ [UNK] của GuwenBERT

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
allc = [c for ch, _ in parsed for c in ch]
unk = sum(1 for c in allc if tokenizer.convert_tokens_to_ids(c) == tokenizer.unk_token_id)
print(f"[UNK] rate trên data train = {unk/max(1,len(allc)):.2%}  (C-CLUE giản thể nên thường thấp)")

## 3. Dựng dataset + tokenize (căn nhãn mức ký tự)

In [ ]:
label_list = ["O"] + sorted({t for _, tg in parsed for t in tg if t != "O"})
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
print("Bộ nhãn model:", label_list)

ds = Dataset.from_dict({"chars": [c for c, _ in parsed], "tags": [t for _, t in parsed]})
ds = ds.train_test_split(test_size=0.1, seed=42)

def encode(batch):
    enc = tokenizer(batch["chars"], is_split_into_words=True, truncation=True, max_length=MAX_LEN)
    labels = []
    for i, tags in enumerate(batch["tags"]):
        wids = enc.word_ids(batch_index=i); prev = None; ids = []
        for wid in wids:
            if wid is None:   ids.append(-100)
            elif wid != prev: ids.append(label2id.get(tags[wid], 0))
            else:             ids.append(-100)
            prev = wid
        labels.append(ids)
    enc["labels"] = labels
    return enc

ds_tok = ds.map(encode, batched=True, remove_columns=ds["train"].column_names)
print(ds_tok)

## 4. Model + Trainer (seqeval)

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID, num_labels=len(label_list), id2label=id2label, label2id=label2id)
collator = DataCollatorForTokenClassification(tokenizer)
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=2)
    tl = [[id2label[l] for pr, l in zip(pp, ll) if l != -100] for pp, ll in zip(preds, p.label_ids)]
    tp = [[id2label[pr] for pr, l in zip(pp, ll) if l != -100] for pp, ll in zip(preds, p.label_ids)]
    r = seqeval.compute(predictions=tp, references=tl)
    return {"precision": r["overall_precision"], "recall": r["overall_recall"], "f1": r["overall_f1"]}

args = TrainingArguments(
    output_dir="guwenbert-ner", learning_rate=3e-5, num_train_epochs=5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    eval_strategy="epoch", save_strategy="epoch", logging_steps=50,
    load_best_model_at_end=True, metric_for_best_model="f1",
    fp16=torch.cuda.is_available(), report_to="none")

import inspect
_tk = dict(model=model, args=args, train_dataset=ds_tok["train"],
           eval_dataset=ds_tok["test"], data_collator=collator, compute_metrics=compute_metrics)
if "processing_class" in inspect.signature(Trainer.__init__).parameters:   # transformers mới
    _tk["processing_class"] = tokenizer
else:
    _tk["tokenizer"] = tokenizer
trainer = Trainer(**_tk)

## 5. Train + đánh giá trên val

In [ ]:
trainer.train()
print("\n=== seqeval trên val ===")
print(trainer.evaluate())

## 6. Suy luận trên corpus của bạn → xuất output
Upload `_seg.csv` (cột `sentence_id, sentence`). Model xuất PER/LOC/ORG/TITLE (tùy nhãn có trong data).

In [ ]:
from google.colab import files
print("Upload _seg.csv (vd HCH_006_001_seg.csv)…")
up = files.upload(); seg_path = next(iter(up))
rows = list(csv.DictReader(open(seg_path, encoding="utf-8-sig")))
print("Số câu:", len(rows))

model.eval()
@torch.no_grad()
def predict(sent):
    chars = [to_simp(c) for c in sent] if INFER_T2S else list(sent)   # 繁->简 nếu cần
    enc = tokenizer(chars, is_split_into_words=True, return_tensors="pt",
                    truncation=True, max_length=MAX_LEN).to(device)
    ids = model(**enc).logits[0].argmax(-1).tolist(); wids = enc.word_ids()
    clab = {}
    for ti, wid in enumerate(wids):
        if wid is not None: clab.setdefault(wid, id2label[ids[ti]])
    tags = [clab.get(i, "O") for i in range(len(sent))]
    ents, i = [], 0
    while i < len(sent):                       # BIO -> span trên câu PHỒN THỂ GỐC (1-1)
        t = tags[i]
        if t.startswith("B-"):
            lab = t[2:]; j = i + 1
            while j < len(sent) and tags[j] == f"I-{lab}": j += 1
            ents.append({"text": sent[i:j], "label": lab}); i = j
        else:
            i += 1
    return ents

out = [[r["sentence_id"], e["text"], e["label"]] for r in rows for e in predict(r["sentence"])]
op = "HCH_ner.finetuned.csv"
with open(op, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f); w.writerow(["sentence_id","text","label"]); w.writerows(out)
print("Đã ghi", len(out), "thực thể ->", op, "| nhãn:", dict(Counter(r[2] for r in out)))
files.download(op)

## 7. (Tuỳ chọn) Đo P/R/F1 trên gold của bạn
Upload gold JSON. Giờ model có PER/LOC/ORG/TITLE → seqeval đo được **PER và LOC** (gold hiện có), so trực tiếp với Hybrid.

In [ ]:
from google.colab import files
print("Upload gold json (vd HCH_006_gold_demo.json)…")
up = files.upload()
gold = json.loads(Path(next(iter(up))).read_text(encoding="utf-8"))["sentences"]

def to_bio(sent, ents):
    tags = ["O"] * len(sent)
    for e in ents:
        idx = sent.find(e["text"])
        if idx < 0: continue
        tags[idx] = "B-" + e["label"]
        for k in range(idx+1, idx+len(e["text"])): tags[k] = "I-" + e["label"]
    return tags

yt = [to_bio(g["sentence"], g["entities"]) for g in gold]
yp = [to_bio(g["sentence"], predict(g["sentence"])) for g in gold]
from seqeval.metrics import classification_report
print(classification_report(yt, yp, digits=3, zero_division=0))

## 8. (Tuỳ chọn) Ghép model fine-tune với Hybrid → đủ 6 nhãn
Upload `_ner.csv` (Hybrid). Lấy **PER/LOC/ORG/TITLE** từ model fine-tune, **giữ TME/NUM** của Hybrid → `HCH_ner.hybrid_ft.csv`.

In [ ]:
from google.colab import files
print("Upload _ner.csv (Hybrid)…")
up = files.upload()
hyb = list(csv.DictReader(open(next(iter(up)), encoding="utf-8-sig")))

FROM_MODEL = {"PER", "LOC", "ORG", "TITLE"}
merged = [[r["sentence_id"], r["text"], r["label"]] for r in hyb if r["label"] not in FROM_MODEL]
merged += [[sid, txt, lab] for sid, txt, lab in out if lab in FROM_MODEL]
with open("HCH_ner.hybrid_ft.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f); w.writerow(["sentence_id","text","label"]); w.writerows(merged)
print("Đã ghi", len(merged), "-> HCH_ner.hybrid_ft.csv")
files.download("HCH_ner.hybrid_ft.csv")

## 9. Lưu model (Google Drive)

In [ ]:
trainer.save_model("guwenbert-ner-final"); tokenizer.save_pretrained("guwenbert-ner-final")
# from google.colab import drive; drive.mount("/content/drive")
# !cp -r guwenbert-ner-final /content/drive/MyDrive/
print("Đã lưu ./guwenbert-ner-final")

---
### Ghi vào report
- **Model:** GuwenBERT fine-tune token-classification trên **C-CLUE** (二十四史, cùng miền) — nhãn PER/LOC/ORG/TITLE.
- **繁/简:** train giản thể → suy luận có OpenCC 繁→简 (span lấy trên chữ phồn thể gốc).
- **Output:** `HCH_ner.finetuned.csv` (4 nhãn ML) và `HCH_ner.hybrid_ft.csv` (đủ 6 nhãn, ghép TME/NUM từ rule).
- **So sánh:** seqeval trên gold + đặt cạnh `_ner.csv` (Hybrid) để chọn phương pháp tốt nhất.
- Bản này **không thể train nhầm trên dữ liệu rỗng** (có GUARD ở cell 2).